# Week 7: Probability → loss functions (MLE, MAP, logistic regression from scratch)

**This week answers the question you raised:** *why* do we use particular loss functions, what MLE and MAP actually mean, and why everything has a log in it.

| Session | Topic | Time |
|---|---|---|
| Day 1 | Probability by simulation; Bayes' rule | 2 h |
| Day 2 | MLE: the parameters that make the data most probable; why logs | 2.5 h |
| Day 3 | MAP: MLE plus a prior, which is the same thing as regularisation | 2.5 h |
| Day 4 | Logistic regression from scratch: sigmoid, cross-entropy, gradient | 2.5 h |
| Review | Spaced recall and a blank-page derivation | 0.5 h |

**Read:** *Mathematics for Machine Learning*, sections 6.1–6.3 and 8.3. **Watch (optional):** StatQuest, "Maximum Likelihood, clearly explained" and "Logistic regression".

### Done when
- [ ] Every test cell prints ✅.
- [ ] On a blank page you can: derive the coin MLE; explain MSE as Gaussian MLE; derive the logistic regression gradient $X^\top(\sigma(Xw) - y)/n$.
- [ ] You can explain in two sentences why L2 regularisation *is* a Gaussian prior.
- [ ] Your logistic regression matches scikit-learn's test accuracy to within 1%.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

---
# Day 1: Probability by simulation (2 h)

When a formula feels abstract, **simulate it**. Code makes probability concrete, and it's how you'll check your derivations from now on.

**Bayes' rule:** $P(D \mid +) = \dfrac{P(+ \mid D)\,P(D)}{P(+)}$, where $P(+) = P(+ \mid D)P(D) + P(+ \mid \neg D)P(\neg D)$.

This is the diagnostic question C5: 1% prevalence, 90% sensitivity, a 5% false-positive rate.

### Exercise 7.1: `posterior_positive`
Return $P(\text{disease} \mid \text{positive test})$.

In [ ]:
def posterior_positive(prevalence, sensitivity, false_pos_rate):
    raise NotImplementedError

In [ ]:
p = posterior_positive(0.01, 0.90, 0.05)
assert abs(p - 0.009 / (0.009 + 0.0495)) < 1e-12
print(f"P(D|+) = {p:.3f}")
print("✅ 7.1 passed")

### Exercise 7.2: Confirm it by simulation
Simulate 1,000,000 people. Each has the disease with probability 1%; then simulate their test results. Estimate $P(D \mid +)$ as (sick and positive) / (positive). Use vectorised NumPy (`rng.random(n) < p`), not a loop.

In [ ]:
def simulate_posterior(prevalence, sensitivity, false_pos_rate, n=1_000_000, seed=0):
    raise NotImplementedError

In [ ]:
est = simulate_posterior(0.01, 0.90, 0.05)
assert abs(est - p) < 0.01
print(f"simulated {est:.3f} vs exact {p:.3f}")
print("✅ 7.2 passed")

**Hints** (open one at a time, and note in your log which ones you needed):

<details><summary>Hint 1</summary>

`sick = rng.random(n) < prevalence`. A test is positive when `(sick & (u < sensitivity)) | (~sick & (u < false_pos_rate))` with `u = rng.random(n)`.

</details>

<details><summary>Hint 2</summary>

The answer is `(sick & positive).sum() / positive.sum()`.

</details>


✍️ *Most positives are false positives here. What does that mean for an AI screening model evaluated only on 'accuracy' or 'sensitivity'? (This is why precision and PPV matter in health AI.)* (double-click to edit)



---
# Day 2: Maximum likelihood estimation (2.5 h)

**The idea in one sentence:** choose the parameters that would have made the data you actually observed **as probable as possible**.

**Coin example:** 10 flips, 7 heads. If $P(\text{heads}) = p$, the probability of this exact sequence is the **likelihood**
$$\mathcal{L}(p) = p^7 (1 - p)^3.$$
Try $p = 0.5$: $0.5^{10} \approx 0.00098$. Try $p = 0.7$: $\approx 0.0022$. So 0.7 explains the data better. Let's find the best $p$.

**Why logs?**
1. A product of many probabilities **underflows** to 0.0 in floating point (the demo below shows it).
2. $\log$ turns products into **sums**, which are far easier to differentiate.
3. $\log$ is increasing, so the $p$ that maximises $\log \mathcal{L}$ also maximises $\mathcal{L}$.

$\log \mathcal{L}(p) = 7 \log p + 3 \log(1 - p)$. Setting the derivative to zero, $\frac{7}{p} - \frac{3}{1-p} = 0$, gives $\hat p = 7/10$. **Do this on paper.**

In [ ]:
probs = np.full(2000, 0.6)
print("product of 2000 probabilities:", np.prod(probs))       # underflow!
print("sum of their logs:", np.log(probs).sum())             # perfectly fine

p_grid = np.linspace(0.01, 0.99, 200)
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(p_grid, p_grid ** 7 * (1 - p_grid) ** 3); ax[0].set_title("likelihood")
ax[1].plot(p_grid, 7 * np.log(p_grid) + 3 * np.log(1 - p_grid)); ax[1].set_title("log-likelihood")
for a in ax: a.axvline(0.7, ls="--", c="gray"); a.set_xlabel("p")
plt.show()

### Exercise 7.3: `bernoulli_log_likelihood`
`data` is an array of 0s and 1s. Return $\sum_i [x_i \log p + (1 - x_i)\log(1 - p)]$. Then the test checks that a grid search finds the maximum at the sample mean.

In [ ]:
def bernoulli_log_likelihood(p, data):
    raise NotImplementedError

In [ ]:
data = (rng.random(500) < 0.3).astype(int)
grid = np.linspace(0.001, 0.999, 999)
best = grid[np.argmax([bernoulli_log_likelihood(p, data) for p in grid])]
assert abs(best - data.mean()) < 0.002
print(f"grid-search MLE {best:.3f} = sample mean {data.mean():.3f}")
print("✅ 7.3 passed")

### The big reveal: loss functions *are* negative log-likelihoods
| If you assume… | then the negative log-likelihood is… | which is the loss called… |
|---|---|---|
| $y \sim \mathcal{N}(f(x), \sigma^2)$ (Gaussian noise) | $\frac{1}{2\sigma^2}\sum (y - f(x))^2 + \text{const}$ | **MSE** (regression) |
| $y \sim \text{Bernoulli}(\sigma(f(x)))$ | $-\sum [y \log \hat p + (1 - y)\log(1 - \hat p)]$ | **binary cross-entropy** |
| $y \sim \text{Categorical}(\text{softmax}(f(x)))$ | $-\sum \log \hat p_{y}$ | **cross-entropy** (multi-class) |

So "minimise MSE" means "find the maximum-likelihood model under Gaussian noise". Choosing a loss is choosing a **probabilistic assumption** about your data. MSE is sensitive to outliers because Gaussian tails are thin. Cross-entropy is used for classification because the labels are Bernoulli or categorical.

✍️ *Derive on paper: the Gaussian log-likelihood of y given prediction f(x), and show that maximising it over f is the same as minimising squared error.* (double-click to edit)



---
# Day 3: MAP, MLE plus a prior (2.5 h)

MLE with little data overfits. Three flips, three heads, gives $\hat p = 1$, which claims tails is *impossible*. **MAP** (maximum a posteriori) adds a **prior belief**:
$$\hat\theta_{MAP} = \arg\max_\theta \; \underbrace{\log P(\text{data} \mid \theta)}_{\text{log-likelihood}} + \underbrace{\log P(\theta)}_{\text{log-prior}}$$

With a $\text{Beta}(a, b)$ prior for the coin ("I've seen roughly $a - 1$ heads and $b - 1$ tails before"):
$$\hat p_{MAP} = \frac{k + a - 1}{n + a + b - 2}.$$

**The connection to regularisation:** put a Gaussian prior $w \sim \mathcal{N}(0, \tau^2 I)$ on a model's weights. Then $\log P(w) = -\frac{1}{2\tau^2}\|w\|^2 + \text{const}$, so
$$\text{MAP} = \arg\min_w \; \text{NLL}(w) + \lambda \|w\|^2, \quad \lambda = \tfrac{1}{2\tau^2}.$$
**L2 regularisation (weight decay) *is* a Gaussian prior.** A strong prior (small $\tau$) means a large $\lambda$: "I believe the weights are small unless the data strongly says otherwise." (Likewise, L1 regularisation is a Laplace prior.)

### Exercise 7.4: `map_coin`

In [ ]:
def map_coin(k, n, a, b):
    raise NotImplementedError

In [ ]:
assert map_coin(7, 10, 1, 1) == 0.7            # a uniform prior gives the MLE
assert abs(map_coin(7, 10, 2, 2) - 8 / 12) < 1e-12
assert abs(map_coin(3, 3, 2, 2) - 0.8) < 1e-12  # no longer claims p = 1
true_p = 0.3
ns = np.arange(1, 400)
flips = rng.random(400) < true_p
ks = np.cumsum(flips)[:len(ns)]
plt.plot(ns, ks / ns, label="MLE", alpha=0.7)
plt.plot(ns, [map_coin(k, n, 10, 10) for k, n in zip(ks, ns)], label="MAP, Beta(10,10) prior")
plt.axhline(true_p, c="k", ls="--"); plt.xscale("log"); plt.xlabel("number of flips"); plt.legend(); plt.show()
print("✅ 7.4 passed")

✍️ *Describe the two curves for small n and for large n. In one sentence: when does the prior matter, and when does it stop mattering? Map this onto 'regularisation matters most with small datasets'.* (double-click to edit)



---
# Day 4: Logistic regression from scratch (2.5 h)

**The model:** $\hat p = \sigma(Xw)$, where $\sigma(z) = 1 / (1 + e^{-z})$ squashes any number into (0, 1), giving a probability. The **sigmoid is the activation function here** because we need a valid probability for a Bernoulli label. (Week 13 covers activations inside networks, which have a different job.)

**The loss:** the Bernoulli negative log-likelihood (binary cross-entropy), averaged:
$$L(w) = -\frac{1}{n}\sum_i \left[y_i \log \hat p_i + (1 - y_i)\log(1 - \hat p_i)\right].$$

**The gradient** (derive it on paper using $\sigma' = \sigma(1 - \sigma)$ from the diagnostic and the chain rule; the terms cancel beautifully):
$$\nabla L = \frac{1}{n} X^\top(\hat p - y).$$
With L2 regularisation (MAP with a Gaussian prior): add $\lambda\|w\|^2$ to the loss and $2\lambda w$ to the gradient. (Usually the intercept isn't regularised; we'll keep it simple and regularise everything.)

### Exercise 7.5: `sigmoid`, `bce_loss`, `bce_grad`
- `sigmoid(z)` must be **numerically stable**: `np.exp(-z)` overflows for z = −1000. Use `np.exp(-np.logaddexp(0, -z))`, which is mathematically identical, or clip z.
- `bce_loss(w, X, y, l2=0.0)` and `bce_grad(w, X, y, l2=0.0)`. Clip $\hat p$ to $[10^{-12}, 1 - 10^{-12}]$ before taking logs.

In [ ]:
def sigmoid(z):
    raise NotImplementedError

def bce_loss(w, X, y, l2=0.0):
    raise NotImplementedError

def bce_grad(w, X, y, l2=0.0):
    raise NotImplementedError

In [ ]:
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("error")                      # any overflow warning fails the test
    assert np.allclose(sigmoid(np.array([-1000.0, 0.0, 1000.0])), [0, 0.5, 1])

def numerical_grad(f, w, h=1e-6):                       # your checker from Week 6
    g = np.zeros_like(w)
    for j in range(len(w)):
        e = np.zeros_like(w); e[j] = h
        g[j] = (f(w + e) - f(w - e)) / (2 * h)
    return g

X = rng.normal(size=(40, 3)); y = (rng.random(40) < 0.5).astype(float); w = rng.normal(size=3)
for l2 in [0.0, 0.1]:
    assert np.allclose(bce_grad(w, X, y, l2), numerical_grad(lambda v: bce_loss(v, X, y, l2), w), atol=1e-6)
print("✅ 7.5 passed: the gradient check is OK")

### Exercise 7.6: Train it on breast cancer data, without leakage
`fit_logreg(X, y, lr=0.1, epochs=2000, l2=0.0)` runs full-batch gradient descent from zeros and returns `w`. Then:
1. Split train/test **before** anything else (stratified, seed 0).
2. Standardise using the **training** mean and std only.
3. Add an intercept column of ones.
4. Train, then compare test accuracy with scikit-learn's `LogisticRegression`.

In [ ]:
def fit_logreg(X, y, lr=0.1, epochs=2000, l2=0.0):
    raise NotImplementedError

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

Xb, yb = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(Xb, yb, test_size=0.25, stratify=yb, random_state=0)
mu, sd = X_tr.mean(axis=0), X_tr.std(axis=0)            # training statistics only
prep = lambda A: np.c_[np.ones(len(A)), (A - mu) / sd]
w = fit_logreg(prep(X_tr), y_tr, l2=0.001)
acc_ours = ((sigmoid(prep(X_te) @ w) > 0.5) == y_te).mean()
acc_sk = LogisticRegression(max_iter=5000).fit((X_tr - mu) / sd, y_tr).score((X_te - mu) / sd, y_te)
print(f"ours {acc_ours:.3f}  sklearn {acc_sk:.3f}")
assert abs(acc_ours - acc_sk) <= 0.01 + 1e-9
print("✅ 7.6 passed")

✍️ *Train with l2 = 0, 0.01, 0.1 and 1.0, and record the test accuracy and ‖w‖. What happens to the weights as l2 grows? Explain it as 'a stronger prior'.* (double-click to edit)



---
# Review (30 min)

### Blank-page derivation (on paper, then photograph it into your repo as `work/week07/derivation.jpg`)
1. The coin MLE, from the likelihood to $\hat p = k/n$.
2. The logistic regression gradient, from the BCE loss to $\frac{1}{n}X^\top(\hat p - y)$.
3. Why L2 regularisation = a Gaussian prior (3 lines).

### Spaced recall
1. *(Week 4)* What two parts does every recursive function need?
2. *(Week 5)* What does `np.linalg.solve(A, b)` compute, and why not use `inv(A) @ b`?
3. *(Week 6)* How do you check a hand-derived gradient?
4. Why do we maximise the *log*-likelihood?
5. MSE corresponds to which noise assumption? And cross-entropy?
6. What happens to MAP as $n \to \infty$?
7. Why must the scaler be fitted on the training data only?

✍️ *Your answer:* (double-click to edit)



### Commit, log, update the tracker